In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Reading

In [0]:
# df = spark.read.format("parquet")\
#                 .load("abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders")

df = spark.read.table("medallion_arc_e2e.bronze.orders")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df = df.withColumnRenamed("_rescued_data", "rescued_data")

In [0]:
df = df.drop("rescued_data")
display(df)

In [0]:
from pyspark.sql.functions import to_timestamp, col
df = df.withColumn("order_date", to_timestamp(col("order_date")))


display(df)

In [0]:
from pyspark.sql.functions import year,month,day
df = df.withColumn("year", year(col("order_date")))\
    .withColumn("month", month(col("order_date")))\
    .withColumn("day", day(col("order_date")))\
    .withColumn("day_name", date_format(col("order_date"), "E"))
display(df)
# df_orders = df.select("order_id", "customer_id", "product_id", "order_date", "quantity", "total_amount")
# display(df_orders)
# df_orders.write.mode("overwrite").saveAsTable("

In [0]:
# df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year","month").orderBy(desc("total_amount"))))
df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

df1.display()

In [0]:
df1 = df1.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
display(df1)

In [0]:
df1 = df1.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
display(df1)

# Classes - OOP

In [0]:
class windows:
    # def __init__(self,df):
    #     self.df = df

    def dense_rank(self,df,partitionBy,orderBy):
        return df.withColumn("flag", dense_rank().over(Window.partitionBy(partitionBy).orderBy(desc(orderBy))))
    
    def rank(self,df,partitionBy,orderBy):
        return df.withColumn("rank_flag", rank().over(Window.partitionBy(partitionBy).orderBy(desc(orderBy))))
    
    def row_number(self,df,partitionBy,orderBy):
        return df.withColumn("row_number_flag", row_number().over(Window.partitionBy(partitionBy).orderBy(desc(orderBy))))
    
    def cume_dist(self,df,partitionBy,orderBy):
        return df.withColumn("cume_dist_flag", cume_dist().over(Window.partitionBy(partitionBy).orderBy(desc(orderBy))))


In [0]:
df_new = df

display(df_new)

In [0]:
obj = windows()

In [0]:
df_new = obj.dense_rank(df_new,"year","total_amount")
# df_new = obj.rank(df_new,"year","total_amount")
# df_new = obj.row_number(df_new,"year","total_amount")
# df_new = obj.cume_dist(df_new,"year","total_amount")

display(df_new)

# Data Writing

In [0]:
df.display()

In [0]:
df.write.format("delta").mode("append")\
                .option("path","abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders")\
                .saveAsTable("medallion_arc_e2e.silver.orders")

# Convert existing path to Delta and register table

> **Error**: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`medallion_arc_e2e`.`silver`.`orders`'). The associated location ('abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders') is not empty and also not a Delta table. SQLSTATE: 42601

If your folder has Parquet/other data, you can turn it into a Delta table:

Python:
```python
from delta.tables import DeltaTable

# Convert to Delta (if path has Parquet files)
DeltaTable.convertToDelta(spark, "parquet.`abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders`")

```

SQL:
```
-- Register as external table in Unity Catalog
spark.sql("""
CREATE TABLE medallion_arc_e2e.silver.orders
USING DELTA
LOCATION 'abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders'
""")
```

In [0]:
# Register as external table in Unity Catalog
spark.sql("""
CREATE TABLE IF NOT EXISTS medallion_arc_e2e.silver.orders
USING DELTA
LOCATION 'abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/orders'
""")